In [1]:
!pip install albumentations -q

In [2]:
from google.colab import drive, files
import os
import shutil

print("="*70)
print("loading Python Code Files")
print("="*70)

# Try to mount Google Drive
try:
    drive.mount('/content/drive', force_remount=True)

    # Path to Python files in Drive
    DRIVE_CODE_DIR = "/content/drive/MyDrive/DeepLearning_HW3/Python_codes"

    code_files = [
        "model_unet.py",
        "config.py",
        "data_loader.py",
        "metrics.py",
        "test_model.py"
    ]

    all_found = True
    for code_file in code_files:
        src = os.path.join(DRIVE_CODE_DIR, code_file)
        if os.path.exists(src):
            shutil.copy(src, f"/content/{code_file}")
            print(f"Copied from Drive: {code_file}")
        else:
            all_found = False
            print(f"Not found in Drive: {code_file}")

    if not all_found:
        print("Some files missing. Please upload manually:")
        uploaded = files.upload()
        print("Files uploaded manually!")

except Exception as e:
    print(f"Drive mount failed: {e}")
    print("Please upload files manually:")
    print("  1. model_unet.py")
    print("  2. config.py")
    print("  3. data_loader.py")
    print("  4. metrics.py")
    print("  5. test_model.py")
    uploaded = files.upload()
    print("Files uploaded!")

print("\n" + "="*70)
print("Python files ready!")
print("="*70)

loading Python Code Files
Mounted at /content/drive
Copied from Drive: model_unet.py
Copied from Drive: config.py
Copied from Drive: data_loader.py
Copied from Drive: metrics.py
Copied from Drive: test_model.py

Python files ready!


In [3]:
import gdown
import os

print("="*70)
print("Loading Best Model (unet4_with_norm.pth)")
print("="*70)

os.makedirs("Models", exist_ok=True)

# Try to download from Google Drive link
model_url = "https://drive.google.com/uc?id=1q3xdQkPVdhLECFBFvUsPNDLhRBc_nsvJ"
model_path = "Models/unet4_with_norm.pth"

try:
    if not os.path.exists(model_path):
        print("Downloading from Google Drive...")
        gdown.download(model_url, model_path, quiet=False)
        print("Model downloaded successfully!")
    else:
        print("Model already exists!")

    # Verify file size
    size_mb = os.path.getsize(model_path) / (1024 * 1024)
    print(f"Model size: {size_mb:.1f} MB")

except Exception as e:
    print(f"Download failed: {e}")
    print("Please upload unet4_with_norm.pth manually:")
    uploaded = files.upload()

    for fname in uploaded.keys():
        if fname.endswith(".pth"):
            shutil.move(fname, model_path)
            print(f"Model uploaded: {fname}")

print("="*70)

Loading Best Model (unet4_with_norm.pth)


Downloading...
From (original): https://drive.google.com/uc?id=1q3xdQkPVdhLECFBFvUsPNDLhRBc_nsvJ
From (redirected): https://drive.google.com/uc?id=1q3xdQkPVdhLECFBFvUsPNDLhRBc_nsvJ&confirm=t&uuid=f686d4cc-f273-4106-a3c1-2eac6827e824
To: /content/Models/unet4_with_norm.pth
100%|██████████| 373M/373M [00:06<00:00, 56.0MB/s]

Model downloaded successfully!
Model size: 355.4 MB


In [4]:
import gdown
import os
import glob

print("="*70)
print("Loading Test Dataset (test.zip)")
print("="*70)

# Try to download test.zip from Google Drive
test_zip_url = "https://drive.google.com/uc?id=1kYdMZ0k6rgnSHwwVnMr7acg1pUjnixdn"
test_zip_path = "test.zip"

try:
    # Check if already extracted
    existing_images = len(glob.glob("/content/Data/test/image/*.png"))

    if existing_images > 0:
        print(f"Test dataset already exists ({existing_images} images)")
    else:
        print("Downloading test.zip from Google Drive...")
        gdown.download(test_zip_url, test_zip_path, quiet=False)

        print("Extracting test.zip...")
        os.system(f"unzip -q '{test_zip_path}' -d /content/Data/")

        # Verify
        test_images = len(glob.glob("/content/Data/test/image/*.png"))
        test_masks = len(glob.glob("/content/Data/test/mask/*.png"))

        print(f"Extracted successfully!")
        print(f"  Images: {test_images}")
        print(f"  Masks: {test_masks}")

except Exception as e:
    print(f"Download failed: {e}")
    print("Please upload test.zip manually:")
    uploaded = files.upload()

    for fname in uploaded.keys():
        if fname.endswith(".zip"):
            print(f"Extracting {fname}...")
            os.system(f"unzip -q '{fname}' -d /content/Data/")
            print("Extracted!")

# Final verification
test_images = len(glob.glob("/content/Data/test/image/*.png"))
test_masks = len(glob.glob("/content/Data/test/mask/*.png"))

print("\n" + "="*70)
print("Test Dataset Ready")
print("="*70)
print(f"  Images: {test_images}")
print(f"  Masks: {test_masks}")
print("="*70)

Loading Test Dataset (test.zip)


Downloading...
From: https://drive.google.com/uc?id=1kYdMZ0k6rgnSHwwVnMr7acg1pUjnixdn
To: /content/test.zip
100%|██████████| 6.89M/6.89M [00:00<00:00, 57.2MB/s]


Extracting test.zip...
Extracted successfully!
  Images: 20
  Masks: 20

Test Dataset Ready
  Images: 20
  Masks: 20


In [5]:
# Fix paths in config.py for Colab environment
with open('config.py', 'r') as f:
    content = f.read()

# Ensure paths are correct for Colab
if 'BASE_DIR' not in content or '/content/Data' not in content:
    # Add/fix BASE_DIR
    if 'BASE_DIR' in content:
        content = content.replace('BASE_DIR = ', '# BASE_DIR = ')

    # Insert correct paths at the beginning
    fixed_content = '''import torch
import os

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Data paths
BASE_DIR = "/content/Data"
TEST_IMG_DIR = os.path.join(BASE_DIR, "test/image")
TEST_MASK_DIR = os.path.join(BASE_DIR, "test/mask")

# Model paths
MODEL_DIR = "/content/Models"

# Model architecture
IN_CHANNELS = 3
OUT_CHANNELS = 1
DROPOUT = 0.1

# Image processing
IMAGE_HEIGHT = 256
IMAGE_WIDTH = 256

''' + content

    with open('config.py', 'w') as f:
        f.write(fixed_content)

print("config.py paths configured for Colab!")

config.py paths configured for Colab!


In [6]:
from test_model import test_model

print("\n" + "="*70)
print("TESTING BEST MODEL: U-Net 4 with Normalization")
print("="*70)

results = test_model(
    model_path='Models/unet4_with_norm.pth',
    model_type='unet4',
    threshold=0.7
)

print("\n" + "="*70)
print("FINAL TEST RESULTS")
print("="*70)
print(f"Mean Dice Score: {results['mean_dice']:.4f}")
print(f"Mean IoU Score:  {results['mean_iou']:.4f}")
print("="*70)

# Expected results
print("Expected Results (from report):")
print("  Dice: ~0.7454")
print("  IoU:  ~0.5946")

# Check if results match
if abs(results['mean_dice'] - 0.7454) < 0.05:
    print("Results match the report! Test successful!")
else:
    print("Results differ from report. Please verify.")

Using device: cpu

TESTING BEST MODEL: U-Net 4 with Normalization
Testing Model
Model: Models/unet4_with_norm.pth
Architecture: unet4
Threshold: 0.7
Device: cpu

Loading model...
Model loaded successfully!

Loading test dataset...
Test samples: 20

Evaluating...


Testing: 100%|██████████| 20/20 [00:43<00:00,  2.17s/it]


Test Results
Mean Dice Score: 0.7454
Mean IoU Score:  0.5946

FINAL TEST RESULTS
Mean Dice Score: 0.7454
Mean IoU Score:  0.5946
Expected Results (from report):
  Dice: ~0.7454
  IoU:  ~0.5946
Results match the report! Test successful!
